# ShotShield v4 — real gunshots from many guns and microphones, and the speaker/room/microphone chain

v3 caught about half of the YouTube gunshots played through a laptop speaker during a live test. An analysis of
those recordings showed three gaps: the speaker removes the bass, rapid fire and dry cracks are rare in the training
data, and the training clips never pass through a loudspeaker, a room and a laptop microphone. v4 keeps the input
format and the network (the browser demo works unchanged) and changes the **data**:

| | v3 | v4 |
|---|---|---|
| Real gunshots | 803 (UrbanSound8K, FSD50K) | + **C3GD** (12,112 shots, 48 guns, 15 mics incl. phones), **multi-firearm** (549 recordings, 4 guns, 27 mics incl. phones, bursts of up to 23 shots), **Belize** (749 shots at 0–1000 m), **M16** (421 shots, 50–100 m, all directions), **SESA** |
| Hard negatives | ESC-50, FSD50K household/impulsive sounds | + **firecrackers** (M16 set), forest background (Belize), explosions/sirens (SESA) |
| Augmentation | reverb, mic EQ, background, clipping | + **loudspeaker playback** (no bass, resonance, limiter), **MP3 compression**, synthetic room with early reflections, laptop microphone; short sounds **placed at random positions** in a background; the same augmentation for gunshots and other sounds |
| Test sets | US8K fold 10, ESC-50 fold 5 + FSD50K eval | + one **unseen recording event** of C3GD, an **unseen session** of the multi-firearm set, Belize's later recording period, M16 and firecrackers from **unseen directions**, SESA's test folder, **simulated speaker playback** of test clips, and your own **demo recordings** if you attach them |

An optional experiment trains a second model **with** in-game PUBG gun sounds (BGG, non-commercial research data)
to see whether game audio helps or hurts on real recordings. The shipped model never uses it.

**Before you run this notebook on Kaggle**
1. **Add Input** → **UrbanSound8K** (the same dataset as v2/v3).
2. Optional: **Add Input** → upload the `.zip` downloaded from the demo's "Save sounds for training" as a private
   dataset. Those clips are used **only for testing**, never for training.
3. **Settings → Accelerator → GPU T4**, **Settings → Internet → On**. About 13 GB of datasets are downloaded
   automatically (Zenodo, Mendeley Data, Hugging Face, GitHub).
4. **Save Version → Save & Run All (Commit)**. It takes about **3–4 hours** (2–3 without the BGG experiment).
5. When it finishes, open the version's **Output** and download **`shotshield_v4_outputs.zip`** and
   **`shotshield_v4_web.zip`**.

In [ ]:
# ONNX conversion tools (tf2onnx without its dependencies: it pins an old protobuf that breaks TensorFlow),
# gdown for the optional BGG download, ffmpeg for MP3 augmentation (normally preinstalled on Kaggle)
!pip install -q onnx onnxruntime gdown
!pip install -q --no-deps tf2onnx
!which ffmpeg || (apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null)

In [ ]:
import os, re, sys, gc, json, time, zlib, shutil, hashlib, zipfile, datetime, subprocess, urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import scipy.signal
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.metrics import roc_auc_score, precision_recall_curve
from sklearn.utils.class_weight import compute_class_weight

# ---- Settings -------------------------------------------------------------
SEED = 42
EPOCHS = 40              # upper bound; early stopping usually ends sooner
BATCH_SIZE = 64
AUG_COPIES = 1           # augmented copies of every training clip
P_PLAYBACK = 0.5         # share of augmented copies that go through the loudspeaker -> room -> mic chain
USE_CODEC = True         # MP3 compression inside the playback chain (needs ffmpeg)
SIM_TEST_PER_CLASS = 1500  # test clips per class re-made as "played through a speaker" (fixed seeds)
RUN_BGG_EXPERIMENT = True  # also train a model with PUBG game gun sounds (comparison only, never shipped)
FSD_CAP = {'dev': 80, 'eval': 30}       # FSD50K clips per hard-negative class (as v3)
FSD_OTHER = {'dev': 800, 'eval': 300}   # plus random other FSD50K clips (speech, music, animals, ...)
BELIZE_BG = {'train': 3000, 'test': 600}  # Belize forest background clips (negatives)

INPUT_DIR = '/kaggle/input'
WORK = '/kaggle/working'
DATA = '/tmp/shotshield_data'          # downloads and caches; /tmp is not part of the notebook output
OUT_DIR = f'{WORK}/shotshield_v4'
WEB_DIR = f'{WORK}/shotshield_v4_web'

REPO_RAW = 'https://raw.githubusercontent.com/nishttw/ShotShield/main'
V3_RELEASE = 'https://github.com/nishttw/ShotShield/releases/download/v3.0'
ESC50_ZIP = 'https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip'
FSD_HF = 'https://huggingface.co/datasets/Fhrozen/FSD50k/resolve/main'

In [ ]:
ASSET_DIR = f'{OUT_DIR}/assets'
for d in (DATA, OUT_DIR, ASSET_DIR, WEB_DIR, f'{WEB_DIR}/examples'):
    os.makedirs(d, exist_ok=True)
rng_global = np.random.default_rng(SEED)
UA = 'Mozilla/5.0 (ShotShield research dataset download)'


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def fetch(url, path, tries=6):
    # Download url to path (skipped if it is already there), retrying with back-off.
    if os.path.exists(path):
        return path
    os.makedirs(os.path.dirname(path), exist_ok=True)
    for i in range(tries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': UA})
            with urllib.request.urlopen(req, timeout=300) as r, open(path + '.part', 'wb') as f:
                shutil.copyfileobj(r, f, 1 << 20)
            os.replace(path + '.part', path)
            return path
        except urllib.error.HTTPError as e:
            if e.code == 404 or i == tries - 1:
                raise
        except Exception:
            if i == tries - 1:
                raise
        time.sleep(min(60, 2 ** i))


# The repository's own code: features (same as v2/v3 and the browser demo), augmentation, windows, datasets
CODE = f'{DATA}/code'
for mod in ('features', 'augment', 'windows', 'sources'):
    p = f'{CODE}/shotshield_{mod}.py'
    if os.path.exists(p):
        os.remove(p)                     # always the current version from GitHub
    fetch(f'{REPO_RAW}/src/shotshield/{mod}.py', p)
sys.path.insert(0, CODE)
os.environ['PYTHONPATH'] = CODE + os.pathsep + os.environ.get('PYTHONPATH', '')   # for joblib workers
import shotshield_features as F
import shotshield_augment as A
import shotshield_windows as W
import shotshield_sources as SRC
CFG = F.FeatureConfig()
SR, N = CFG.sample_rate, CFG.n_samples
print('librosa', librosa.__version__, '| ffmpeg:', A.FFMPEG)
print('disk free in /tmp: %.0f GB' % (shutil.disk_usage('/tmp').free / 1e9))

## 1. Data
### UrbanSound8K, ESC-50, FSD50K (as in v3)
UrbanSound8K keeps its official folds: 1–8 train, 9 validation, **10 test**. ESC-50 folds 1–3 train, 4 validation,
**5 test**. FSD50K: every gunshot clip plus capped hard-negative classes; dev-train / dev-val / **eval = test**.

In [ ]:
csv_hits, fold1_hits = [], []
for root, dirs, files in os.walk(INPUT_DIR, followlinks=True):
    csv_hits += [os.path.join(root, f) for f in files if f.lower() == 'urbansound8k.csv']
    fold1_hits += [os.path.join(root, d) for d in dirs if d.lower() == 'fold1']
if not csv_hits or not fold1_hits:
    raise FileNotFoundError('UrbanSound8K not found. Click "Add Input", search "UrbanSound8K", add it, then run again.')

meta = pd.read_csv(sorted(csv_hits)[0])
audio_root = os.path.dirname(sorted(fold1_hits)[0])
us8k = pd.DataFrame({
    'source': 'us8k', 'name': meta['slice_file_name'],
    'path': [os.path.join(audio_root, f'fold{f}', n) for f, n in zip(meta['fold'], meta['slice_file_name'])],
    'group': meta['fsID'].astype(str), 'fsid': meta['fsID'].astype(int),
    'label': (meta['class'] == 'gun_shot').astype(int), 'category': meta['class'],
    'kind': np.where(meta['class'] == 'gun_shot', 'gunshot', 'urban'), 'fold': meta['fold'].astype(int),
    'split': meta['fold'].map(lambda f: 'test' if f == 10 else 'val' if f == 9 else 'train'),
    'rank': 0, 'license': 'CC BY-NC 3.0', 'credit': 'UrbanSound8K, J. Salamon, C. Jacoby, J. P. Bello'})
print(f'UrbanSound8K: {len(us8k)} clips, {us8k.label.sum()} gunshots')

ESC_IMPULSIVE = {'clapping', 'door_wood_knock', 'mouse_click', 'keyboard_typing', 'can_opening', 'glass_breaking',
                 'footsteps', 'clock_tick', 'fireworks', 'sneezing', 'coughing', 'water_drops', 'thunderstorm',
                 'crackling_fire', 'door_wood_creaks'}
ESC_MACHINE = {'washing_machine', 'vacuum_cleaner', 'engine', 'train', 'airplane', 'helicopter', 'chainsaw',
               'hand_saw', 'toilet_flush', 'clock_alarm'}
BACKGROUND = ['rain', 'sea_waves', 'wind', 'crickets', 'insects', 'chirping_birds', 'pouring_water', 'crackling_fire',
              'washing_machine', 'vacuum_cleaner', 'engine', 'train', 'airplane', 'helicopter']

esc_csv = [os.path.join(r, f) for r, _, fs in os.walk(INPUT_DIR, followlinks=True) for f in fs if f.lower() == 'esc50.csv']
if esc_csv:
    esc_root = os.path.dirname(os.path.dirname(sorted(esc_csv)[0]))
else:
    zpath = fetch(ESC50_ZIP, f'{DATA}/esc50.zip')
    if not os.path.isdir(f'{DATA}/ESC-50-master'):
        with zipfile.ZipFile(zpath) as z:
            z.extractall(DATA)
    esc_csv, esc_root = [f'{DATA}/ESC-50-master/meta/esc50.csv'], f'{DATA}/ESC-50-master'
esc_meta = pd.read_csv(sorted(esc_csv)[0])
wav_index = {f: os.path.join(r, f) for r, _, fs in os.walk(esc_root, followlinks=True) for f in fs if f.endswith('.wav')}
esc = pd.DataFrame({
    'source': 'esc50', 'name': esc_meta['filename'], 'path': esc_meta['filename'].map(wav_index),
    'group': esc_meta['src_file'].astype(str), 'fsid': esc_meta['src_file'].astype(int), 'label': 0,
    'category': esc_meta['category'],
    'kind': ['impulsive' if c in ESC_IMPULSIVE else 'machine' if c in ESC_MACHINE else 'other' for c in esc_meta['category']],
    'split': esc_meta['fold'].map({1: 'train', 2: 'train', 3: 'train', 4: 'val', 5: 'test'}),
    'rank': 0, 'license': 'CC BY-NC 3.0', 'credit': 'ESC-50, K. J. Piczak'}).dropna(subset=['path'])
print(f'ESC-50: {len(esc)} clips')

In [ ]:
FSD_IMPULSIVE = [
    'Finger_snapping', 'Clapping', 'Slam', 'Knock', 'Door', 'Sliding_door', 'Drawer_open_or_close', 'Cupboard_open_or_close',
    'Dishes_and_pots_and_pans', 'Cutlery_and_silverware', 'Chink_and_clink', 'Glass', 'Shatter', 'Crack', 'Thump_and_thud',
    'Tap', 'Hammer', 'Wood', 'Tick', 'Coin_(dropping)', 'Keys_jangling', 'Computer_keyboard', 'Typing', 'Typewriter',
    'Camera', 'Scissors', 'Ratchet_and_pawl', 'Fireworks', 'Boom', 'Explosion', 'Thunder', 'Snare_drum', 'Bass_drum',
    'Drum', 'Hi-hat', 'Cymbal', 'Tabla', 'Writing', 'Packing_tape_and_duct_tape', 'Tearing', 'Crumpling_and_crinkling',
    'Crushing', 'Zipper_(clothing)', 'Walk_and_footsteps', 'Squeak', 'Mechanisms', 'Tools', 'Power_tool', 'Drill',
    'Sawing', 'Engine_starting', 'Cough', 'Sneeze', 'Splash_and_splatter']
FSD_MACHINE = [
    'Mechanical_fan', 'Microwave_oven', 'Printer', 'Engine', 'Idling', 'Accelerating_and_revving_and_vroom', 'Motorcycle',
    'Car_passing_by', 'Truck', 'Traffic_noise_and_roadway_noise', 'Buzz', 'Hiss', 'Boiling', 'Frying_(food)',
    'Water_tap_and_faucet', 'Toilet_flush', 'Rain', 'Wind', 'Rail_transport', 'Aircraft']
GUN = 'Gunshot_and_gunfire'

for f in ('dev.csv', 'eval.csv', 'vocabulary.csv'):
    fetch(f'{FSD_HF}/labels/{f}', f'{DATA}/fsd/{f}')
fetch(f'{FSD_HF}/metadata/eval_clips_info_FSD50K.json', f'{DATA}/fsd/eval_info.json')
vocab = set(pd.read_csv(f'{DATA}/fsd/vocabulary.csv', header=None)[1])
assert not [c for c in FSD_IMPULSIVE + FSD_MACHINE + [GUN] if c not in vocab]


def fsd_select(df, part):
    df = df.assign(L=df['labels'].str.split(','))
    is_gun = df['L'].apply(lambda l: GUN in l)
    picked = [df[is_gun].assign(category=GUN, kind='gunshot')]
    neg = df[~is_gun]
    taken = set()
    has = {c: neg['L'].apply(lambda l, c=c: c in l) for c in FSD_IMPULSIVE + FSD_MACHINE}
    for c in sorted(has, key=lambda c: has[c].sum()):          # rare classes first
        pool = neg[has[c] & ~neg.index.isin(taken)]
        pool = pool.sample(min(FSD_CAP[part], len(pool)), random_state=SEED)
        taken.update(pool.index)
        picked.append(pool.assign(category=c, kind='impulsive' if c in FSD_IMPULSIVE else 'machine'))
    rest = neg[~neg.index.isin(taken)]
    picked.append(rest.sample(min(FSD_OTHER[part], len(rest)), random_state=SEED).assign(category='other', kind='other'))
    out = pd.concat(picked)
    return pd.DataFrame({
        'source': 'fsd50k', 'name': out['fname'].astype(str) + '.wav',
        'path': [f'{DATA}/fsd/{part}/{f}.wav' for f in out['fname']],
        'group': out['fname'].astype(str).values, 'fsid': out['fname'].astype(int).values,
        'label': (out['kind'] == 'gunshot').astype(int).values, 'category': out['category'].values,
        'kind': out['kind'].values, 'split': out['split'].values if part == 'dev' else 'test', 'rank': 0,
        'license': 'per clip (Creative Commons)', 'credit': 'FSD50K, E. Fonseca et al.',
        'url': [f'{FSD_HF}/clips/{part}/{f}.wav' for f in out['fname']]})


fsd = pd.concat([fsd_select(pd.read_csv(f'{DATA}/fsd/dev.csv'), 'dev'),
                 fsd_select(pd.read_csv(f'{DATA}/fsd/eval.csv'), 'eval')], ignore_index=True)
esc = esc[~esc.fsid.isin(us8k.fsid)]                                  # all three come from Freesound
fsd = fsd[~fsd.fsid.isin(us8k.fsid) & ~fsd.fsid.isin(esc.fsid)]


def download(row):
    try:
        fetch(row.url, row.path)
        sf.info(row.path)
        return True
    except Exception as e:
        try:
            os.remove(row.path)
        except OSError:
            pass
        return repr(e)


t0 = time.time()
with ThreadPoolExecutor(16) as pool:
    ok = np.array([r is True for r in pool.map(download, fsd.itertuples())])
fsd = fsd[ok].drop(columns='url')
print(f'FSD50K: {ok.sum()} clips ready, {(~ok).sum()} failed — {time.time() - t0:.0f}s')

### New in v4: open gunshot datasets
Each loader downloads its dataset (CC BY 4.0) and returns the same columns. Splits keep everything that belongs
together (the same shot on several microphones, one recording session) on one side. A dataset that fails to
download is skipped and reported instead of stopping the run.

In [ ]:
LOADERS = {
    'c3gd': lambda: SRC.load_c3gd(DATA),
    'multi_firearm': lambda: SRC.load_multi_firearm(DATA, seed=SEED),
    'belize': lambda: SRC.load_belize(DATA, BELIZE_BG['train'], BELIZE_BG['test'], seed=SEED),
    'm16': lambda: SRC.load_m16(DATA),
    'sesa': lambda: SRC.load_sesa(DATA),
}
new_sets, failed = [], {}
for name, load in LOADERS.items():
    t0 = time.time()
    try:
        df = load()
        new_sets.append(df)
        print(f'{name:14s} {len(df):6d} clips  ({df.label.sum()} gunshots)  {time.time() - t0:5.0f}s')
    except Exception as e:
        failed[name] = repr(e)
        print(f'{name:14s} FAILED: {e!r}')
print('disk free in /tmp: %.0f GB' % (shutil.disk_usage('/tmp').free / 1e9))

home = SRC.load_home_clips(INPUT_DIR)
print(f'your demo recordings: {len(home)} clips ({int(home.label.sum()) if len(home) else 0} gunshots) — test only')

bgg = None
if RUN_BGG_EXPERIMENT:
    try:
        bgg = SRC.load_bgg(DATA)
        print(f'BGG (PUBG game audio): {len(bgg)} clips, {int(bgg.label.sum())} gun sounds')
    except Exception as e:
        failed['bgg'] = repr(e)
        print('BGG download failed, experiment skipped:', repr(e))

In [ ]:
clips = pd.concat([us8k, esc, fsd] + new_sets + ([home] if len(home) else []), ignore_index=True)
clips['rank'] = clips['rank'].fillna(0).astype(int)
# a file that fails to load later is dropped; quick check now that every path exists
missing = ~clips['path'].map(os.path.exists)
if missing.any():
    print('missing files dropped:', clips[missing].groupby('source').size().to_dict())
    clips = clips[~missing].reset_index(drop=True)
summary = clips.groupby(['source', 'split', 'label']).size().unstack(fill_value=0).rename(columns={0: 'other', 1: 'gunshots'})
display(summary)
print(f"total {len(clips)} clips, {clips.label.sum()} gunshots ({clips.label.mean():.1%})")

## 2. Windows, augmentation, features
* **Digital silence is trimmed** (as in v3).
* **Clean copy:** the loudest 4 s of a recording (the k-th loudest for the extra firecracker windows).
  Sounds shorter than 2.4 s (all of C3GD, many UrbanSound8K/FSD50K clips) are **placed once at a random position
  over a quiet background** (15–35 dB below the sound). v3 looped them, which turns a 1 s gunshot into four
  identical shots exactly 1 s apart, a pattern real audio never has. Clips of 2.4–4 s are still loop-padded.
  Both rules apply to gunshots and other sounds alike, so neither becomes a shortcut.
* **Augmented copy** (training clips only): the event moves to a random position in the window (short clips are
  mostly placed into a background sound instead of looped); then, half of the time, the **playback chain**: MP3 at
  32–128 kbit/s, a small loudspeaker (bass cut at 80–500 Hz, a resonance, a limiter), a room and a laptop
  microphone. Otherwise a direct room/microphone recording. Then background sounds, noise and clipping. The bass
  loss was matched to 27 YouTube gunshots recorded through a laptop.
* **Simulated playback test:** up to 1,500 test gunshots and 1,500 test non-gunshots are re-made through the
  playback chain with fixed seeds. They are never trained on.

In [ ]:
# Backgrounds: steady sounds from training data only (ESC-50 folds 1-3, FSD50K dev-train machine noise)
bank = pd.concat([esc[(esc.split == 'train') & esc.category.isin(BACKGROUND)]['path'],
                  fsd[(fsd.split == 'train') & (fsd.kind == 'machine')]['path']]).values
print(len(bank), 'background clips')


def load_bg(path, rng):
    b = W.trim_digital_silence(F.load_audio(path, CFG))
    if len(b) == 0:
        return None
    b = b[rng.integers(0, max(1, len(b) - N)):]
    return F.loop_pad(b, N)


SHORT = int(0.6 * N)   # sounds shorter than 2.4 s are placed into a background instead of being looped


def placed(y, rng, bg_path):
    # A short sound once, at a random position, over a quiet background: what the live window sees.
    # (Looping a 1 s gunshot would make 4 identical shots exactly 1 s apart, a pattern real audio never has.)
    bg = load_bg(bg_path, rng) if isinstance(bg_path, str) else None
    if bg is None:
        bg = scipy.signal.lfilter(A.PINK_B, A.PINK_A, rng.standard_normal(N))
    return A.place_event(y, N, rng, bg=bg, snr_db=rng.uniform(15, 35))


def row_features(path, rank=0, seed=-1, bg_path=None, p_playback=P_PLAYBACK):
    # One clip -> (128, 173) float16 log-mel. seed >= 0 makes an augmented copy.
    try:
        y = W.trim_digital_silence(F.load_audio(path, CFG))
        if len(y) == 0:
            return None
        start = W.kth_loudest_start(y, N, int(rank))
        if start is None:
            return None                                   # no k-th event in this recording
        if seed < 0:                                      # clean copy (deterministic per file)
            if len(y) < SHORT:
                key = zlib.crc32(f'{path}#{rank}'.encode())
                win = placed(y, np.random.default_rng(key), bank[key % len(bank)] if len(bank) else None)
            else:
                win = F.loop_pad(y, N) if len(y) <= N else y[start:start + N]
            return F.log_mel(win, CFG).astype(np.float16)
        rng = np.random.default_rng(seed)
        bg = load_bg(bg_path, rng) if isinstance(bg_path, str) else None
        if len(y) <= N:
            if rng.random() < (0.75 if len(y) < SHORT else 0.5):
                win, bg = placed(y, rng, bg_path), None      # the background is used up
            else:
                win = F.loop_pad(y, N)                       # as the browser does for short uploaded files
        else:
            win = W.random_event_crop(y, N, start, rng)
        win = A.augment(win, SR, rng, bg=bg, p_playback=p_playback, use_codec=USE_CODEC)
        return F.log_mel(win, CFG).astype(np.float16)
    except Exception as e:
        print('skipped', os.path.basename(path), '-', repr(e)[:120])
        return None


def build_rows(df, aug_copies, sim_per_class, seed_base):
    clean = df.assign(aug='clean', seed=-1, p_play=P_PLAYBACK)
    parts = [clean]
    train = df[df.split == 'train']
    for c in range(aug_copies):
        parts.append(train.assign(aug='train_aug', seed=seed_base + c * 10_000_000 + np.arange(len(train)),
                                  p_play=P_PLAYBACK))
    test = df[(df.split == 'test') & (df.source != 'home')]
    sim = pd.concat([test[test.label == k].sample(min(sim_per_class, (test.label == k).sum()), random_state=SEED)
                     for k in (0, 1)]) if sim_per_class else test.iloc[0:0]
    parts.append(sim.assign(aug='sim_playback', seed=seed_base + 90_000_000 + np.arange(len(sim)), p_play=1.0))
    rows = pd.concat(parts, ignore_index=True)
    r = np.random.default_rng(seed_base)
    rows['bg'] = [bank[r.integers(len(bank))] if a != 'clean' and r.random() < 0.6 else None for a in rows['aug']]
    return rows


def compute_features(rows, tag):
    cache = f'{DATA}/features_{tag}_{len(rows)}.npy'
    if os.path.exists(cache):
        X_ = np.load(cache)
        ok_ = np.load(cache.replace('.npy', '_ok.npy'))
        print('loaded cached features', cache)
    else:
        t0 = time.time()
        feats = Parallel(n_jobs=-1, batch_size=16, verbose=1)(
            delayed(row_features)(p, k, s, b, pp)
            for p, k, s, b, pp in zip(rows['path'], rows['rank'], rows['seed'], rows['bg'], rows['p_play']))
        ok_ = np.array([f is not None for f in feats])
        X_ = np.stack([f for f in feats if f is not None])
        del feats
        np.save(cache, X_)
        np.save(cache.replace('.npy', '_ok.npy'), ok_)
        print(f'{tag}: {len(rows)} rows in {(time.time() - t0) / 60:.0f} min')
    return X_, ok_


rows = build_rows(clips, AUG_COPIES, SIM_TEST_PER_CLASS, SEED)
X, ok = compute_features(rows, 'v4')
dropped = rows[~ok].groupby(['source', 'aug']).size()
rows = rows[ok].reset_index(drop=True)
y = rows['label'].values.astype(np.float32)
print(f'X {X.shape} {X.dtype}, {X.nbytes / 1e9:.1f} GB, {int(y.sum())} gunshot rows; dropped: {dropped.to_dict()}')

In [ ]:
# Clean vs augmented examples, to check that augmentation looks sensible
pairs = []
for src, cat in [('c3gd', None), ('multi_firearm', None), ('m16', 'firecracker'), ('fsd50k', 'Finger_snapping'),
                 ('us8k', 'gun_shot')]:
    cand = rows[(rows.source == src) & (rows.aug == 'train_aug') & ((rows.category == cat) if cat else True)]
    if len(cand):
        r = cand.iloc[0]
        clean = rows.index[(rows.path == r.path) & (rows['rank'] == r['rank']) & (rows.aug == 'clean')]
        if len(clean):
            pairs.append((f'{src} {cat or ""}', clean[0], cand.index[0]))
if pairs:
    fig, axes = plt.subplots(2, len(pairs), figsize=(4 * len(pairs), 6), squeeze=False)
    for j, (title, a, b) in enumerate(pairs):
        for i, (idx, kind) in enumerate([(a, 'original'), (b, 'augmented')]):
            axes[i, j].imshow(X[idx].astype(np.float32), origin='lower', aspect='auto', cmap='magma', vmin=0, vmax=1)
            axes[i, j].set_title(f'{title} — {kind}', fontsize=9); axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
    plt.tight_layout(); plt.savefig(f'{ASSET_DIR}/augmentation_v4.png', dpi=110); plt.show()

## 3. Model
The same network as v2/v3 (so the browser demo needs no changes): three convolution blocks, global average + max
pooling, 304,289 parameters. Trained on all training rows; early-stopped and threshold-tuned on the validation rows
of every dataset.

In [ ]:
import tensorflow as tf
import keras
from keras import layers
keras.utils.set_random_seed(SEED)
print('TensorFlow', tf.__version__, '| Keras', keras.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))


def build_model(input_shape=(CFG.n_mels, CFG.n_frames, 1)):
    inp = keras.Input(shape=input_shape, name='log_mel')
    x = inp
    for filters in (32, 64, 128):
        for _ in range(2):
            x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            x = layers.ReLU()(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(0.2)(x)
    x = layers.Concatenate()([layers.GlobalAveragePooling2D()(x), layers.GlobalMaxPooling2D()(x)])
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid', name='gunshot_probability')(x)
    model = keras.Model(inp, out, name='shotshield_v4')
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy',
                  metrics=[keras.metrics.AUC(curve='PR', name='pr_auc'), keras.metrics.AUC(name='auc')])
    return model


class Batches(keras.utils.PyDataset):
    # float16 features -> float32 batches, so the training set never exists twice in float32.
    # With class weights, each batch also carries per-sample weights (gunshots are the minority in most sets).
    def __init__(self, Xs, ys, idx, batch_size, shuffle, class_weight=None, seed=SEED, **kw):
        super().__init__(**kw)
        self.Xs, self.ys, self.idx, self.bs, self.shuffle = Xs, ys, np.asarray(idx), batch_size, shuffle
        self.cw = class_weight
        self.rng = np.random.default_rng(seed)
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.idx) / self.bs))

    def on_epoch_end(self):
        self.order = self.rng.permutation(self.idx) if self.shuffle else self.idx

    def __getitem__(self, i):
        b = np.sort(self.order[i * self.bs:(i + 1) * self.bs])
        xb, yb = self.Xs[b].astype(np.float32)[..., None], self.ys[b]
        if self.cw is None:
            return xb, yb
        return xb, yb, np.where(yb > 0.5, self.cw[1], self.cw[0]).astype(np.float32)


def train(Xs, ys, train_idx, val_idx, verbose=2):
    keras.backend.clear_session()
    model = build_model()
    w = compute_class_weight('balanced', classes=np.array([0.0, 1.0]), y=ys[train_idx])
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_pr_auc', mode='max', patience=6, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_pr_auc', mode='max', factor=0.5, patience=2, min_lr=1e-5)]
    history = model.fit(Batches(Xs, ys, train_idx, BATCH_SIZE, True, class_weight=w),
                        validation_data=Batches(Xs, ys, val_idx, 256, False),
                        epochs=EPOCHS, callbacks=callbacks, verbose=verbose)
    return model, history


def predict(model, Xs, idx):
    idx = np.asarray(idx)
    if len(idx) == 0:
        return np.zeros(0)
    return np.concatenate([model.predict(Xs[c].astype(np.float32)[..., None], batch_size=256, verbose=0).ravel()
                           for c in np.array_split(idx, int(np.ceil(len(idx) / 4096)))])


def best_f1_threshold(yt, p):
    pr, rc, t = precision_recall_curve(yt, p)
    f1 = 2 * pr[:-1] * rc[:-1] / np.maximum(pr[:-1] + rc[:-1], 1e-9)
    return float(t[np.argmax(f1)])


def threshold_for_recall(yt, p, target):
    pos = np.sort(p[yt == 1])
    return float(pos[int(np.floor((1 - target) * len(pos)))]) if len(pos) else 0.5


AUG = rows['aug'].values
SPLIT = rows['split'].values
TRAIN = np.where((SPLIT == 'train') & (AUG != 'sim_playback'))[0]
VAL = np.where((SPLIT == 'val') & (AUG == 'clean'))[0]
print(f'train {len(TRAIN)} rows ({int(y[TRAIN].sum())} gunshot), validation {len(VAL)} ({int(y[VAL].sum())} gunshot)')

## 4. Train v4

In [ ]:
model, history = train(X, y, TRAIN, VAL)
val_prob = predict(model, X, VAL)
THRESHOLD = best_f1_threshold(y[VAL], val_prob)
print(f'threshold (best F1 on validation): {THRESHOLD:.3f}')
for target in (0.9, 0.95, 0.98):
    t = threshold_for_recall(y[VAL], val_prob, target)
    print(f'  for {target:.0%} validation recall: threshold {t:.3f}, validation false alarms '
          f'{(val_prob[y[VAL] == 0] >= t).mean():.2%}')

## 5. Test sets: v3 vs v4
Every test set is scored with the released **v3** model and with **v4**, each at its own threshold. "Simulated
playback" is the test clips re-made through the loudspeaker/room/microphone chain.

In [ ]:
TEST_SETS = {}
for src in ['us8k', 'esc50', 'fsd50k', 'c3gd', 'multi_firearm', 'belize', 'm16', 'sesa']:
    idx = np.where((rows.source.values == src) & (SPLIT == 'test') & (AUG == 'clean'))[0]
    if len(idx):
        TEST_SETS[src] = idx
TEST_SETS['simulated playback'] = np.where(AUG == 'sim_playback')[0]
hard = np.where((SPLIT == 'test') & (AUG == 'clean') & np.isin(rows.source.values, ['esc50', 'fsd50k']))[0]
TEST_SETS['hard negatives (ESC-50 + FSD50K)'] = hard[y[hard] == 0]
fc = np.where((SPLIT == 'test') & (AUG == 'clean') & (rows.category.values == 'firecracker'))[0]
if len(fc):
    TEST_SETS['firecrackers'] = fc

v3_model, V3_THRESHOLD = None, None
try:
    v3_info = json.load(open(fetch(f'{V3_RELEASE}/model_info.json', f'{DATA}/v3/model_info.json')))
    v3_path = fetch(f'{V3_RELEASE}/shotshield_v3.keras', f'{DATA}/v3/shotshield_v3.keras')
    assert sha256(v3_path) == v3_info['model_sha256'], 'v3 checksum mismatch'
    v3_model, V3_THRESHOLD = keras.models.load_model(v3_path), v3_info['threshold']
except Exception as e:
    print('v3 model not available, comparison skipped:', repr(e))


def score_sets(model_, thr):
    out = {}
    for name, idx in TEST_SETS.items():
        p = predict(model_, X, idx)
        yt = y[idx]
        r = {'clips': len(idx), 'gunshots': int(yt.sum())}
        if yt.sum():
            r['gunshots caught'] = f'{int((p[yt == 1] >= thr).sum())}/{int(yt.sum())} ({(p[yt == 1] >= thr).mean():.1%})'
        if (yt == 0).any():
            r['false alarms'] = f'{int((p[yt == 0] >= thr).sum())}/{int((yt == 0).sum())} ({(p[yt == 0] >= thr).mean():.1%})'
        if 0 < yt.sum() < len(yt):
            r['ROC-AUC'] = round(roc_auc_score(yt, p), 4)
        out[name] = (r, p)
    return out


v4_scores = score_sets(model, THRESHOLD)
v3_scores = score_sets(v3_model, V3_THRESHOLD) if v3_model else {}
table = []
for name in TEST_SETS:
    for tag, sc, thr in [('v3', v3_scores, V3_THRESHOLD), ('v4', v4_scores, THRESHOLD)]:
        if name in sc:
            table.append({'test set': name, 'model': f'{tag} (threshold {thr:.2f})', **sc[name][0]})
results = pd.DataFrame(table).set_index(['test set', 'model'])
display(results.fillna(''))

In [ ]:
# False alarms by kind of sound (all clean test negatives), and gunshots caught by dataset
neg_idx = np.where((SPLIT == 'test') & (AUG == 'clean') & (y == 0) & (rows.source.values != 'home'))[0]
fa = rows.loc[neg_idx, ['source', 'category']].assign(v4=predict(model, X, neg_idx) >= THRESHOLD)
if v3_model:
    fa['v3'] = predict(v3_model, X, neg_idx) >= V3_THRESHOLD
by_cat = fa.groupby(['source', 'category'])[[c for c in ('v3', 'v4') if c in fa]].agg(['sum', 'count'])
by_cat.columns = [f'{m} {k}' for m, k in by_cat.columns]
by_cat = by_cat.rename(columns={'v4 count': 'clips'}).drop(columns=[c for c in by_cat if c == 'v3 count'])
by_cat.to_csv(f'{OUT_DIR}/test_false_alarms_by_class.csv')
print('Sounds v4 still confuses most:')
display(by_cat.sort_values('v4 sum', ascending=False).head(15))

In [ ]:
# Your demo recordings: scored like the live page (4 s windows every 0.25 s, highest window counts)
home_table = None
if len(home):
    def live_max(model_, path):
        yy = F.load_audio(path, CFG)
        fe, _ = F.windows_to_features(yy, CFG, 0.25)
        return float(model_.predict(fe, verbose=0).max())
    home_table = home[['name', 'label', 'note']].copy()
    home_table['v4'] = [live_max(model, p) for p in home.path]
    if v3_model:
        home_table['v3'] = [live_max(v3_model, p) for p in home.path]
    display(home_table.round(3))
    for tag, thr in [('v3', V3_THRESHOLD), ('v4', THRESHOLD)]:
        if tag in home_table:
            g = home_table[home_table.label == 1][tag]
            n = home_table[home_table.label == 0][tag]
            print(f'{tag}: gunshots caught {int((g >= thr).sum())}/{len(g)}, other sounds flagged {int((n >= thr).sum())}/{len(n)}')

## 6. Experiment: does game audio help? (BGG, PUBG gun sounds)
A second model is trained on the same data **plus** 80% of BGG and scored on the same test sets, plus the held-back
20% of BGG. It is only a comparison: BGG is non-commercial research data, so it never goes into the shipped model.

In [ ]:
bgg_results = None
if RUN_BGG_EXPERIMENT and bgg is not None and len(bgg):
    bgg_rows = build_rows(bgg, AUG_COPIES, 0, SEED + 7)
    Xb, okb = compute_features(bgg_rows, 'bgg')
    bgg_rows = bgg_rows[okb].reset_index(drop=True)
    yb = bgg_rows['label'].values.astype(np.float32)
    X_all = np.concatenate([X, Xb])
    y_all = np.concatenate([y, yb])
    off = len(X)
    tr_b = off + np.where(bgg_rows.split.values == 'train')[0]
    te_b = off + np.where((bgg_rows.split.values == 'test') & (bgg_rows.aug.values == 'clean'))[0]
    model_b, _ = train(X_all, y_all, np.concatenate([TRAIN, tr_b]), VAL, verbose=0)
    thr_b = best_f1_threshold(y[VAL], predict(model_b, X_all, VAL))
    rows_b = []
    for name, idx in list(TEST_SETS.items()) + [('BGG game sounds (held back)', te_b)]:
        for tag, m_, thr, Xs, ys in [('v4', model, THRESHOLD, X_all, y_all), ('v4 + BGG', model_b, thr_b, X_all, y_all)]:
            p = predict(m_, Xs, idx)
            yt = ys[idx]
            rows_b.append({'test set': name, 'model': f'{tag} (threshold {thr:.2f})',
                           'gunshots caught': f'{(p[yt == 1] >= thr).mean():.1%}' if yt.sum() else '',
                           'false alarms': f'{(p[yt == 0] >= thr).mean():.1%}' if (yt == 0).any() else ''})
    bgg_results = pd.DataFrame(rows_b).set_index(['test set', 'model'])
    display(bgg_results)
    del X_all, model_b; gc.collect()
else:
    print('BGG experiment skipped')

## 7. Save and package

In [ ]:
MODEL_PATH = f'{OUT_DIR}/shotshield_v4.keras'
model.save(MODEL_PATH)
reloaded = keras.models.load_model(MODEL_PATH)
some = TEST_SETS['c3gd'][:32] if 'c3gd' in TEST_SETS else VAL[:32]
assert np.allclose(predict(reloaded, X, some), predict(model, X, some), atol=1e-5), 'reload mismatch'

counts = clips.groupby(['source', 'split', 'label']).size()
info = {
    'name': 'ShotShield', 'version': '4.0',
    'created': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'task': 'binary: does a 4 s audio window contain a gunshot?',
    'labels': {'0': 'not_gunshot', '1': 'gun_shot'},
    'threshold': THRESHOLD,
    'features': {'sample_rate': SR, 'clip_seconds': CFG.clip_seconds, 'n_mels': CFG.n_mels, 'n_fft': CFG.n_fft,
                 'hop_length': CFG.hop_length, 'top_db': CFG.top_db, 'n_frames': CFG.n_frames, 'padding': 'loop'},
    'input_shape': [CFG.n_mels, CFG.n_frames, 1],
    'parameters': int(model.count_params()),
    'training': {
        'datasets': {s: {sp: {'other': int(counts.get((s, sp, 0), 0)), 'gunshot': int(counts.get((s, sp, 1), 0))}
                         for sp in ('train', 'val', 'test')} for s in clips.source.unique()},
        'failed_downloads': failed,
        'windows': 'digital silence trimmed; clean: loudest 4 s (k-th loudest for extra firecracker events), loop-pad if < 4 s; '
                   'augmented: event at a random position, short clips placed into a background',
        'augmentation': {'copies': AUG_COPIES, 'p_playback': P_PLAYBACK, 'codec': bool(USE_CODEC and A.FFMPEG),
                         'chain': 'MP3 -> loudspeaker (HP 80-500 Hz, resonance, limiter) -> room -> laptop mic; '
                                  'or room/mic; + background, noise, clipping'},
        'threshold': 'best F1 on the validation rows of all datasets',
        'epochs_run': len(history.history['loss']), 'class_weighting': 'balanced'},
    'tests': {name: {'v4': v4_scores[name][0], 'v3': v3_scores.get(name, (None,))[0]} for name in TEST_SETS},
    'home_recordings': None if home_table is None else home_table.drop(columns='note').round(4).to_dict('records'),
    'credits': {k: v for k, v in SRC.CREDITS.items() if k != 'bgg'},
    'versions': {'tensorflow': tf.__version__, 'keras': keras.__version__, 'librosa': librosa.__version__},
    'model_sha256': sha256(MODEL_PATH),
}
json.dump(info, open(f'{OUT_DIR}/model_info.json', 'w'), indent=2, default=str)
clips.drop(columns=['path']).to_csv(f'{OUT_DIR}/data_manifest_v4.csv', index=False)
results.to_csv(f'{OUT_DIR}/test_results_v3_vs_v4.csv')
if bgg_results is not None:
    bgg_results.to_csv(f'{OUT_DIR}/bgg_experiment.csv')
if home_table is not None:
    home_table.to_csv(f'{OUT_DIR}/home_recordings.csv', index=False)
pred_rows = rows.loc[np.unique(np.concatenate(list(TEST_SETS.values()) + [VAL])),
                     ['source', 'name', 'category', 'split', 'aug', 'label']]
pred_rows['v4'] = predict(model, X, pred_rows.index.values)
if v3_model:
    pred_rows['v3'] = predict(v3_model, X, pred_rows.index.values)
pred_rows.to_csv(f'{OUT_DIR}/predictions.csv', index=False)
print(json.dumps(info['tests'], indent=1, default=str)[:3000])

## 8. Browser bundle
ONNX conversion as in v3, checked against Keras. Example clips: the v3 picks plus test clips from the new datasets
(C3GD, multi-firearm burst, firecracker), with their credits.

In [ ]:
CONVERT = r'''
import os, sys
import numpy as np
strategy, model_path, out_path = sys.argv[1:4]
import keras
model = keras.models.load_model(model_path)
model(np.zeros((1,) + tuple(model.inputs[0].shape[1:]), 'float32'))   # export needs a model that has been called
if strategy == 'tf2onnx':
    import tensorflow as tf, tf2onnx
    spec = (tf.TensorSpec((None,) + tuple(model.inputs[0].shape[1:]), tf.float32, name='log_mel'),)
    fn = tf.function(lambda x: model(x, training=False), input_signature=spec)
    tf2onnx.convert.from_function(fn, input_signature=spec, opset=17, output_path=out_path)
else:
    model.export(out_path, format='onnx')
'''

examples, waves = [], {}
fold10 = clips[(clips.source == 'us8k') & (clips.split == 'test')]
picks = [fold10[fold10.category == 'gun_shot'].sample(2, random_state=7)]
for cls in ['dog_bark', 'siren', 'drilling', 'jackhammer', 'air_conditioner', 'car_horn']:
    picks.append(fold10[fold10.category == cls].sample(1, random_state=7))
lic_ok = ('publicdomain/zero', 'licenses/by/', 'licenses/by-nc/')
fsd_info = json.load(open(f'{DATA}/fsd/eval_info.json'))
fsd_test = clips[(clips.source == 'fsd50k') & (clips.split == 'test') & (clips.label == 0)]
for cls in ['Finger_snapping', 'Clapping', 'Knock', 'Slam', 'Dishes_and_pots_and_pans', 'Typing', 'Fireworks', 'Mechanical_fan']:
    cand = fsd_test[(fsd_test.category == cls) & fsd_test.fsid.astype(int).astype(str).map(
        lambda f: any(k in fsd_info.get(f, {}).get('license', '') for k in lic_ok))]
    cand = cand[[sf.info(p).duration <= 8 for p in cand.path]]
    if len(cand):
        picks.append(cand.sample(1, random_state=7))
for src, cond in [('c3gd', lambda d: d.is_phone == 1), ('c3gd', lambda d: d.is_phone == 0),
                  ('multi_firearm', lambda d: d.num_gunshots >= 4), ('m16', lambda d: d.category == 'firecracker')]:
    d = clips[(clips.source == src) & (clips.split == 'test')]
    if len(d) and src in ('c3gd', 'multi_firearm', 'm16'):
        d = d[cond(d)]
        if len(d):
            picks.append(d.sample(1, random_state=7))
chosen = pd.concat(picks)

DOI = {'c3gd': 'https://doi.org/10.48550/arXiv.2606.18135', 'multi_firearm': 'https://doi.org/10.5281/zenodo.7004819',
       'm16': 'https://doi.org/10.17632/kbgw323248.1'}
for _, r in chosen.iterrows():
    stem = os.path.splitext(os.path.basename(str(r['name'])))[0][:40]
    name = re.sub(r'[^A-Za-z0-9_.-]+', '-', f"{r['category']}_{stem}").strip('-')
    y_ex = W.trim_digital_silence(F.load_audio(r['path'], CFG))
    if len(y_ex) > 8 * SR:                      # long recordings: 6 s around the (k-th) loudest event
        s = W.kth_loudest_start(y_ex, N, int(r['rank'])) or 0
        a = max(0, s - SR)
        y_ex = y_ex[a:a + 6 * SR]
    wav = f'{WEB_DIR}/examples/{name}.wav'
    sf.write(wav, np.clip(np.round(y_ex * 32767), -32768, 32767).astype(np.int16), SR, subtype='PCM_16')
    waves[name], _ = sf.read(wav, dtype='float32')      # what the browser decodes
    e = {'name': name, 'file': f'examples/{name}.wav', 'class': str(r['category']).split(':')[-1].lower(),
         'is_gunshot': bool(r['label']), 'duration': round(len(waves[name]) / SR, 3), 'dataset': r['source'],
         'slice_file_name': str(r['name']), 'note': None, 'fold': 10 if r['source'] == 'us8k' else None}
    if r['source'] in ('us8k', 'fsd50k'):
        e.update(fsID=int(r['fsid']), source=f"https://freesound.org/s/{int(r['fsid'])}/")
    if r['source'] == 'fsd50k':
        meta_ = fsd_info.get(str(int(r['fsid'])), {})
        e.update(note='FSD50K eval set (never trained on)',
                 credit={'title': meta_.get('title'), 'uploader': meta_.get('uploader'), 'license': meta_.get('license')})
    elif r['source'] in DOI:
        e.update(note=f"{r['source']} test set (never trained on)", source=DOI[r['source']],
                 credit={'title': str(r['name']), 'uploader': SRC.CREDITS[r['source']].split(',')[0] + ' dataset',
                         'license': 'https://creativecommons.org/licenses/by/4.0/'})
    examples.append(e)

rng_ref = np.random.default_rng(2026)
t_ref = np.arange(N) / SR
shots = np.zeros(N)
for s in (0.4, 1.6, 1.75, 3.1):
    i = int(s * SR); shots[i:i + SR // 2] += 0.9 * rng_ref.standard_normal(SR // 2) * np.exp(-np.arange(SR // 2) / (0.03 * SR))
waves.update({'synth_white_noise': (0.3 * rng_ref.standard_normal(N)).astype(np.float32),
              'synth_impulses_in_noise': (shots + 0.01 * rng_ref.standard_normal(N)).astype(np.float32),
              'synth_chirp': (0.5 * np.sin(2 * np.pi * 50 * 4 / np.log(200) * (200 ** (t_ref / 4) - 1))).astype(np.float32)})
arrays, feats_all, probs_all = {}, [], []
for name, wv in waves.items():
    fe, st = F.windows_to_features(wv, CFG)
    pr = model.predict(fe, verbose=0).ravel()
    arrays.update({f'wave/{name}': wv, f'feat/{name}': fe, f'starts/{name}': np.asarray(st), f'prob/{name}': pr.astype(np.float32)})
    feats_all.append(fe); probs_all.append(pr)
for e in examples:
    pr = arrays[f"prob/{e['name']}"]
    e.update(keras_window_probs=[round(float(v), 6) for v in pr], keras_max_prob=round(float(pr.max()), 6),
             keras_detected=bool(pr.max() >= THRESHOLD))
np.savez_compressed(f'{WEB_DIR}/reference.npz', info=json.dumps({'librosa': librosa.__version__, 'keras': keras.__version__,
                    'threshold': THRESHOLD, 'model': 'v4'}), **arrays)
X_ref, p_keras = np.concatenate(feats_all).astype(np.float32), np.concatenate(probs_all)

import onnx, onnxruntime as ort
chosen_onnx = None
for strategy in ['tf2onnx', 'keras_tf']:
    path = f'{DATA}/v4_{strategy}.onnx'
    r = subprocess.run([sys.executable, '-c', CONVERT, strategy, MODEL_PATH, path], capture_output=True, text=True, timeout=1200)
    if r.returncode or not os.path.exists(path):
        print(strategy, 'failed:', (r.stderr or r.stdout).strip().splitlines()[-1:])
        continue
    m = onnx.load(path)
    sess = ort.InferenceSession(m.SerializeToString(), providers=['CPUExecutionProvider'])
    inp = sess.get_inputs()[0]
    p_onnx = np.concatenate([sess.run(None, {inp.name: X_ref[i:i + 1]})[0].ravel() for i in range(len(X_ref))])
    err = float(np.abs(p_onnx - p_keras).max())
    print(f'{strategy}: max |ONNX − Keras| = {err:.2e} over {len(X_ref)} windows')
    if err < 1e-4:
        chosen_onnx = (strategy, m, inp, err)
        break
assert chosen_onnx, 'ONNX conversion failed — send the output above'
strategy, onnx_model, inp, err = chosen_onnx
onnx.save(onnx_model, f'{WEB_DIR}/shotshield_v4.onnx')

web_info = dict(info)
web_info['web'] = {'onnx_file': 'shotshield_v4.onnx', 'onnx_sha256': sha256(f'{WEB_DIR}/shotshield_v4.onnx'),
                   'converter': strategy, 'input_name': inp.name, 'input_shape': str(inp.shape),
                   'opset': max(o.version for o in onnx_model.opset_import if o.domain in ('', 'ai.onnx')),
                   'max_abs_diff_vs_keras': err, 'checked_windows': int(len(X_ref)),
                   'versions': {'onnx': onnx.__version__, 'onnxruntime': ort.__version__}}
json.dump(web_info, open(f'{WEB_DIR}/model_info.json', 'w'), indent=2, default=str)
json.dump({'description': 'UrbanSound8K test fold 10 (CC BY-NC 3.0), FSD50K eval set (per-clip Creative Commons '
                          'licenses), C3GD, multi-firearm and M16/firecracker test sets (CC BY 4.0)',
           'threshold': THRESHOLD, 'examples': examples}, open(f'{WEB_DIR}/examples/examples.json', 'w'), indent=2)
display(pd.DataFrame(examples)[['name', 'is_gunshot', 'duration', 'keras_max_prob', 'keras_detected']])

for d in (OUT_DIR, WEB_DIR):
    z = shutil.make_archive(d + '_outputs' if d == OUT_DIR else d, 'zip', d)
    print(f'{z}: {os.path.getsize(z) / 1e6:.1f} MB, sha256 {sha256(z)}')

## 9. Results for the README

In [ ]:
def as_markdown(df):
    try:
        return df.to_markdown()
    except ImportError:          # tabulate missing
        return df.to_string()


print(as_markdown(results.fillna('')))
if bgg_results is not None:
    print(); print(as_markdown(bgg_results))
print(f"\nParameters: {model.count_params():,} | threshold: {THRESHOLD:.3f} | failed downloads: {failed or 'none'}")

In [ ]:
# Stop the background worker processes so the run can end cleanly
from joblib.externals.loky import get_reusable_executor
get_reusable_executor().shutdown(wait=True)
print('DONE — download shotshield_v4_outputs.zip and shotshield_v4_web.zip from /kaggle/working')